# Sparse autoencoders — automatically recovering features from a model

Step 5 of 6 in the mech interp curriculum.

We close the loop on the whole curriculum. Step 1 showed that models pack many features into fewer dimensions (superposition); the features become directions in activation space rather than individual neurons. This makes interpretation hard. In this notebook we train a sparse autoencoder on a superposition model's hidden activations and watch the SAE recover the original feature directions — without ever being told what they are.

Read `README.md` first. It explains what an SAE is and how the experiment is structured.

Expected runtime: ~3 minutes, no GPU required.

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

torch.manual_seed(0)
np.random.seed(0)

## 2. Train a superposition model (the "base model")

Same setup as step 1, but with 10 features into 5 hidden dimensions instead of 5 into 2. At high sparsity, this forces meaningful superposition.

The ground-truth feature directions are the columns of `W`. We'll use these as the reference the SAE has to recover.

In [ ]:
N_FEATURES = 10
N_HIDDEN   = 5
SPARSITY   = 0.9

class ToyModel(nn.Module):
    def __init__(self, n_features, n_hidden):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_hidden, n_features))
        nn.init.xavier_normal_(self.W)
        self.b = nn.Parameter(torch.zeros(n_features))

    def encode(self, x):                      # x: (B, n_features) → (B, n_hidden)
        return x @ self.W.T

    def forward(self, x):
        return F.relu(self.encode(x) @ self.W + self.b)

def make_sparse_batch(B, n_features, sparsity, device):
    vals = torch.rand(B, n_features, device=device)
    mask = torch.rand(B, n_features, device=device) > sparsity
    return vals * mask

def importance(n_features, device):
    return 0.7 ** torch.arange(n_features, device=device, dtype=torch.float32)

base = ToyModel(N_FEATURES, N_HIDDEN).to(device)
imp  = importance(N_FEATURES, device)
opt  = torch.optim.AdamW(base.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10_000)

for step in range(10_000):
    x = make_sparse_batch(1024, N_FEATURES, SPARSITY, device)
    loss = (((base(x) - x) ** 2) * imp).mean()
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    if step % 2000 == 0:
        print(f'  base step {step:5d}  loss {loss.item():.5f}')

print('Base model trained.')
ground_truth = base.W.detach().clone()   # (n_hidden, n_features) -- columns are feature dirs
print(f'Ground-truth feature matrix W shape: {tuple(ground_truth.shape)}')

## 3. Collect hidden activations from the base model

These are what we'll train the SAE on. We sample 50,000 sparse inputs and push them through the base model's encoder, getting a `(50_000, 5)` tensor.

In [ ]:
with torch.no_grad():
    big_x  = make_sparse_batch(50_000, N_FEATURES, SPARSITY, device)
    big_h  = base.encode(big_x)              # (50_000, 5)

print(f'Hidden activation dataset: {tuple(big_h.shape)}')
print(f'Mean abs activation: {big_h.abs().mean().item():.3f}')

## 4. The sparse autoencoder

Implementation:

- Encoder: `Linear(din, dsae) → ReLU`
- Decoder: `Linear(dsae, din)` (no bias output bias, but with a learnable pre-encoder bias offset on the input — a standard trick from Anthropic's SAE paper)
- Loss: reconstruction MSE + λ × L1 norm of hidden activations

We also constrain decoder columns to unit norm after each step — without this, the SAE can cheat the L1 penalty by shrinking hidden activations while growing decoder weights.

In [ ]:
D_SAE   = 20      # 4x overcomplete
L1_COEF = 1e-3

class SAE(nn.Module):
    def __init__(self, d_in, d_sae):
        super().__init__()
        self.W_enc = nn.Parameter(torch.randn(d_in, d_sae) * (1 / d_in ** 0.5))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.randn(d_sae, d_in) * (1 / d_sae ** 0.5))
        # tie decoder columns to unit norm by re-normalising the parameter at init
        with torch.no_grad():
            self.W_dec.data /= self.W_dec.data.norm(dim=1, keepdim=True)
        self.b_pre = nn.Parameter(torch.zeros(d_in))  # pre-encoder bias trick

    def forward(self, x):
        centred = x - self.b_pre
        h       = F.relu(centred @ self.W_enc + self.b_enc)
        x_hat   = h @ self.W_dec + self.b_pre
        return x_hat, h

    def normalize_decoder(self):
        with torch.no_grad():
            norms = self.W_dec.data.norm(dim=1, keepdim=True).clamp(min=1e-8)
            self.W_dec.data /= norms

## 5. Train the SAE

Standard loop. We log reconstruction loss and L0 (the average number of active features per input — features with `h > 0`). A healthy L0 for our high-sparsity inputs is around 1, matching the average number of input features that were on.

In [ ]:
sae       = SAE(N_HIDDEN, D_SAE).to(device)
opt       = torch.optim.AdamW(sae.parameters(), lr=3e-4)
n_steps   = 8000
batch_sz  = 1024

recon_history, l0_history = [], []

for step in range(n_steps):
    idx = torch.randint(0, big_h.shape[0], (batch_sz,), device=device)
    x   = big_h[idx]

    x_hat, h = sae(x)
    recon = ((x_hat - x) ** 2).mean()
    l1    = h.abs().sum(dim=-1).mean()
    loss  = recon + L1_COEF * l1

    opt.zero_grad(); loss.backward(); opt.step()
    sae.normalize_decoder()

    if step % 200 == 0:
        with torch.no_grad():
            l0 = (h > 0).float().sum(dim=-1).mean().item()
        recon_history.append(recon.item())
        l0_history.append(l0)
        if step % 1000 == 0:
            print(f'  sae step {step:5d}  recon {recon.item():.5f}  L0 {l0:.2f}')

print('SAE trained.')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(recon_history); axes[0].set_yscale('log')
axes[0].set_title('reconstruction loss'); axes[0].grid(True, alpha=0.3)
axes[1].plot(l0_history)
axes[1].set_title('L0 — average # active features per input'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Compare SAE features to ground-truth features

Each row of `sae.W_dec` is a direction in 5-dim space — one of the SAE's learnt features. Each column of `ground_truth` is also a direction in 5-dim space — one of the original model's ground-truth features.

We compute the cosine similarity between every pair: a `(dsae, nfeatures) = (20, 10)` matrix. The story we want: for every column, at least one row has cosine ≈ 1.

In [ ]:
with torch.no_grad():
    sae_dirs = sae.W_dec.detach()                            # (d_sae=20, d_in=5)
    sae_unit = sae_dirs / sae_dirs.norm(dim=1, keepdim=True).clamp(min=1e-8)

    gt_dirs  = ground_truth.T                                # (n_features=10, d_in=5)
    gt_unit  = gt_dirs / gt_dirs.norm(dim=1, keepdim=True).clamp(min=1e-8)

    cos = sae_unit @ gt_unit.T                               # (20, 10)
    cos_np = cos.cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 8))
vmax = np.abs(cos_np).max()
im = ax.imshow(cos_np, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
ax.set_xlabel('ground-truth feature i')
ax.set_ylabel('SAE feature j')
ax.set_xticks(range(N_FEATURES)); ax.set_yticks(range(D_SAE))
for i in range(D_SAE):
    for j in range(N_FEATURES):
        if abs(cos_np[i, j]) > 0.5:
            ax.text(j, i, f'{cos_np[i, j]:.2f}', ha='center', va='center', fontsize=7, color='black')
plt.colorbar(im, label='cosine similarity')
ax.set_title('SAE features vs ground-truth features — cosine similarity\n'
             '(one bright cell per column ⇒ SAE recovered that feature)')
plt.tight_layout(); plt.show()

# best match per ground-truth feature
print('Best SAE match for each ground-truth feature:')
for i in range(N_FEATURES):
    best_sae = int(cos_np[:, i].argmax())
    print(f'  GT feat {i}: best SAE feat = {best_sae:2d}  (cos = {cos_np[best_sae, i]:.3f})')

## 7. Visualise the matched feature directions

For each ground-truth feature, find its best-matching SAE feature and plot the two directions side by side. They should look nearly identical (since we're plotting unit vectors).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x_pos = np.arange(N_HIDDEN)
width = 0.08
colors = plt.cm.tab10(np.linspace(0, 1, N_FEATURES))

for i in range(N_FEATURES):
    best_sae = int(cos_np[:, i].argmax())
    gt = gt_unit[i].cpu().numpy()
    # align sign
    sae_dir = sae_unit[best_sae].cpu().numpy()
    if np.dot(sae_dir, gt) < 0:
        sae_dir = -sae_dir
    ax.plot(x_pos + (i - N_FEATURES / 2) * width, gt,      'o-', color=colors[i], alpha=0.9)
    ax.plot(x_pos + (i - N_FEATURES / 2) * width, sae_dir, 'x',  color=colors[i], alpha=0.9, markersize=8)

ax.set_xlabel('hidden dimension'); ax.set_ylabel('component value (unit vector)')
ax.set_xticks(x_pos)
ax.set_title('Ground-truth features (o lines) vs best-match SAE features (x marks) — they overlap')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Discussion

What just happened. We trained a model that packs 10 features into 5 hidden dimensions via superposition. Then we trained an SAE on its 5-dimensional hidden activations. With no knowledge of the ground-truth features, just an L1 sparsity penalty, the SAE recovered the original feature directions. Cosine similarity between matched pairs should be ≥ 0.95 for every ground-truth feature.

Why this matters. This is the central technique of modern interpretability. The same procedure — train an SAE on a real model's hidden activations, look at what features it learns — is what produced the Golden Gate Bridge feature in Anthropic's Scaling Monosemanticity. At scale, SAEs recover thousands of clean features from real LLMs.

Caveats (which you'll hit immediately if you try this on a real model).

- Hyperparameter sensitivity. The L1 coefficient is the single most important knob. Too small → SAE doesn't learn sparse features; too big → loses reconstruction. Tuning matters.
- Dead features. Some SAE features may end up always-zero. Standard mitigation: resampling — periodically reinitialise dead features.
- Feature splitting. A wider SAE often splits what looks like one feature into several finer ones. Is this revealing real structure or overfitting? Active research.
- Two SAEs ≠ same features. Train two SAEs on the same data with different seeds and you'll get slightly different dictionaries. Is there a canonical decomposition? Open question.

Where to next. This is the end of the curriculum. You've done every major move in mech interp. See the top-level `README.md` for what to do from here — open problems, communities, frontier reading. Realistically: pick one open problem and try a small experiment of your own.